# Projeto 3 — CNNs para classificação de cenas aéreas (RESISC45)

Este notebook define as **arquiteturas convolucionais** do projeto, escritas do zero em PyTorch (sem pesos pré-treinados), **treina as oito CNNs** com os mesmos hiperparâmetros das MLPs (seção 3) e depois **busca hiperparâmetros com o Optuna** e as treina de novo (seções 4 e 5). A seção 7 é o bônus com YOLO. São duas famílias, cada uma em quatro variantes, para comparar o efeito de **módulos de atenção** com custo computacional semelhante.

| Família | Variantes | Parâmetros |
|---|---|---|
| ResNet-18 | `ResNet18`, `ResNet18SE`, `ResNet18CBAM`, `ResNet18SECBAM` | 11,18 M a 11,36 M |
| VGG-16 (com BatchNorm) | `VGG16`, `VGG16SE`, `VGG16CBAM`, `VGG16SECBAM` | 14,72 M a 14,88 M |

**Decisões comuns a todos os modelos**

- Entrada RGB e 10 classes (ordem fixa das classes definida no projeto).
- **Agregação global** (média espacial) antes da camada de classificação. Assim a quantidade de parâmetros é **a mesma em 128 × 128 e em 256 × 256**, como exige o experimento de resolução.
- Treino sempre a partir de pesos novos.
- Cada classe é independente: o código dos blocos se repete de propósito, para que cada arquitetura possa ser lida isoladamente.
- Os parâmetros foram contados instanciando cada modelo (apenas pesos treináveis).

### Módulos de atenção usados

- **SE (Squeeze-and-Excitation)** — calcula a média espacial de cada canal, passa por duas camadas densas (redução 16) e uma sigmoide, e usa o resultado para **ponderar os canais**. Referência: Hu et al. (2018).
- **CBAM (Convolutional Block Attention Module)** — aplica atenção de **canal** (média e máximo espacial por um MLP compartilhado) e, em seguida, atenção **espacial** (convolução 7 × 7 sobre a média e o máximo dos canais). Referência: Woo et al. (2018).
- **SE + CBAM** — o SE é aplicado primeiro e o CBAM depois. A ordem foi uma escolha nossa e deve ser registrada no artigo. Como o SE e a atenção de canal do CBAM têm função parecida, pode haver redundância.

## 1. Família ResNet-18

ResNet-18 (He et al., 2016): *stem* 7 × 7 com MaxPool, quatro estágios (64 → 128 → 256 → 512 canais) com dois blocos residuais cada e atalhos de identidade (convolução 1 × 1 quando muda a resolução ou o número de canais). Nas variantes com atenção, o módulo é aplicado **em cada bloco residual, antes da soma com o atalho**.

| Classe | Atenção | Parâmetros | GFLOPs (128) | GFLOPs (256) |
|---|---|---|---|---|
| `ResNet18` | nenhuma | 11.181.642 | 1,18 | 4,74 |
| `ResNet18SE` | SE | 11.270.722 | 1,18 | 4,74 |
| `ResNet18CBAM` | CBAM | 11.271.506 | 1,19 | 4,74 |
| `ResNet18SECBAM` | SE + CBAM | 11.360.586 | 1,19 | 4,74 |

No fim da célula de código abaixo há um **teste de sanidade**: ele imprime os parâmetros de cada modelo e confirma que a saída é `(N, 10)` em 128 × 128 e em 256 × 256.

In [ ]:
import torch
from torch import nn


class ResNet18(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(3, 2, 1))
        self.blocos, self.atalhos = nn.ModuleList(), nn.ModuleList()
        c_in = 64
        for c, s in [(64, 1), (64, 1), (128, 2), (128, 1), (256, 2), (256, 1), (512, 2), (512, 1)]:
            self.blocos.append(nn.Sequential(
                nn.Conv2d(c_in, c, 3, s, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(),
                nn.Conv2d(c, c, 3, 1, 1, bias=False), nn.BatchNorm2d(c)))
            self.atalhos.append(
                nn.Sequential(nn.Conv2d(c_in, c, 1, s, bias=False), nn.BatchNorm2d(c)) if s != 1 or c_in != c else nn.Identity())
            c_in = c
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        x = self.stem(x)
        for bloco, atalho in zip(self.blocos, self.atalhos):
            x = torch.relu(bloco(x) + atalho(x))
        return self.fc(x.mean((2, 3)))  # agregação global: mesmos parâmetros em 128 e 256


class ResNet18SE(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(3, 2, 1))
        self.blocos, self.atalhos, self.ses = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        c_in = 64
        for c, s in [(64, 1), (64, 1), (128, 2), (128, 1), (256, 2), (256, 1), (512, 2), (512, 1)]:
            self.blocos.append(nn.Sequential(
                nn.Conv2d(c_in, c, 3, s, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(),
                nn.Conv2d(c, c, 3, 1, 1, bias=False), nn.BatchNorm2d(c)))
            self.ses.append(nn.Sequential(nn.Linear(c, c // 16), nn.ReLU(), nn.Linear(c // 16, c), nn.Sigmoid()))
            self.atalhos.append(
                nn.Sequential(nn.Conv2d(c_in, c, 1, s, bias=False), nn.BatchNorm2d(c)) if s != 1 or c_in != c else nn.Identity())
            c_in = c
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        x = self.stem(x)
        for bloco, atalho, se in zip(self.blocos, self.atalhos, self.ses):
            y = bloco(x)
            y = y * se(y.mean((2, 3)))[:, :, None, None]  # SE: pondera cada canal
            x = torch.relu(y + atalho(x))
        return self.fc(x.mean((2, 3)))



class ResNet18CBAM(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(3, 2, 1))
        self.blocos, self.atalhos, self.mlps, self.espaciais = nn.ModuleList(), nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        c_in = 64
        for c, s in [(64, 1), (64, 1), (128, 2), (128, 1), (256, 2), (256, 1), (512, 2), (512, 1)]:
            self.blocos.append(nn.Sequential(
                nn.Conv2d(c_in, c, 3, s, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(),
                nn.Conv2d(c, c, 3, 1, 1, bias=False), nn.BatchNorm2d(c)))
            self.mlps.append(nn.Sequential(nn.Linear(c, c // 16), nn.ReLU(), nn.Linear(c // 16, c)))
            self.espaciais.append(nn.Conv2d(2, 1, 7, padding=3, bias=False))
            self.atalhos.append(
                nn.Sequential(nn.Conv2d(c_in, c, 1, s, bias=False), nn.BatchNorm2d(c)) if s != 1 or c_in != c else nn.Identity())
            c_in = c
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        x = self.stem(x)
        for bloco, atalho, mlp, espacial in zip(self.blocos, self.atalhos, self.mlps, self.espaciais):
            y = bloco(x)
            y = y * torch.sigmoid(mlp(y.mean((2, 3))) + mlp(y.amax((2, 3))))[:, :, None, None]  # CBAM: atenção de canal
            y = y * torch.sigmoid(espacial(torch.cat([y.mean(1, keepdim=True), y.amax(1, keepdim=True)], 1)))  # CBAM: atenção espacial
            x = torch.relu(y + atalho(x))
        return self.fc(x.mean((2, 3)))


class ResNet18SECBAM(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(3, 2, 1))
        self.blocos, self.atalhos, self.ses, self.mlps, self.espaciais = nn.ModuleList(), nn.ModuleList(), nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        c_in = 64
        for c, s in [(64, 1), (64, 1), (128, 2), (128, 1), (256, 2), (256, 1), (512, 2), (512, 1)]:
            self.blocos.append(nn.Sequential(
                nn.Conv2d(c_in, c, 3, s, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(),
                nn.Conv2d(c, c, 3, 1, 1, bias=False), nn.BatchNorm2d(c)))
            self.ses.append(nn.Sequential(nn.Linear(c, c // 16), nn.ReLU(), nn.Linear(c // 16, c), nn.Sigmoid()))
            self.mlps.append(nn.Sequential(nn.Linear(c, c // 16), nn.ReLU(), nn.Linear(c // 16, c)))
            self.espaciais.append(nn.Conv2d(2, 1, 7, padding=3, bias=False))
            self.atalhos.append(
                nn.Sequential(nn.Conv2d(c_in, c, 1, s, bias=False), nn.BatchNorm2d(c)) if s != 1 or c_in != c else nn.Identity())
            c_in = c
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        x = self.stem(x)
        for bloco, atalho, se, mlp, espacial in zip(self.blocos, self.atalhos, self.ses, self.mlps, self.espaciais):
            y = bloco(x)
            y = y * se(y.mean((2, 3)))[:, :, None, None]  # SE: pondera cada canal
            y = y * torch.sigmoid(mlp(y.mean((2, 3))) + mlp(y.amax((2, 3))))[:, :, None, None]  # CBAM: atenção de canal
            y = y * torch.sigmoid(espacial(torch.cat([y.mean(1, keepdim=True), y.amax(1, keepdim=True)], 1)))  # CBAM: atenção espacial
            x = torch.relu(y + atalho(x))
        return self.fc(x.mean((2, 3)))


for Modelo in (ResNet18, ResNet18SE, ResNet18CBAM, ResNet18SECBAM):
    m = Modelo()
    print(Modelo.__name__, f"{sum(p.numel() for p in m.parameters()):,} parâmetros",
          m(torch.randn(2, 3, 128, 128)).shape, m(torch.randn(2, 3, 256, 256)).shape)

## 2. Família VGG-16

VGG-16 (Simonyan e Zisserman, 2015) com **BatchNorm**: 13 convoluções 3 × 3 em cinco blocos (64 → 128 → 256 → 512 → 512), cada um terminando em MaxPool 2 × 2. As camadas densas originais foram substituídas por agregação global e uma única camada linear de saída. Nas variantes com atenção, o módulo é aplicado **no fim de cada bloco, antes do MaxPool** (5 módulos por rede).

| Classe | Atenção | Parâmetros | Tamanho (MB, float32) | GFLOPs (128) | GFLOPs (256) |
|---|---|---|---|---|---|
| `VGG16` | nenhuma | 14.724.042 | 56,2 | 10,02 | 40,09 |
| `VGG16SE` | SE | 14.801.894 | 56,5 | 10,02 | 40,09 |
| `VGG16CBAM` | CBAM | 14.802.384 | 56,5 | 10,03 | 40,11 |
| `VGG16SECBAM` | SE + CBAM | 14.880.236 | 56,8 | 10,03 | 40,11 |

A VGG-16 realiza cerca de **8 vezes mais operações** que a ResNet-18, o que pesa sobretudo em 256 × 256. A atenção acrescenta menos de 1% de parâmetros e custo praticamente nulo em operações.

O fim da célula de código abaixo repete o mesmo teste de sanidade (parâmetros e saída `(N, 10)` nas duas resoluções).

In [ ]:
import torch
from torch import nn


class VGG16(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.blocos = nn.ModuleList()
        c_in = 3
        for canais in [[64, 64], [128, 128], [256, 256, 256], [512, 512, 512], [512, 512, 512]]:
            camadas = []
            for c in canais:
                camadas += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU()]
                c_in = c
            self.blocos.append(nn.Sequential(*camadas))
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        for bloco in self.blocos:
            x = bloco(x)
            x = nn.functional.max_pool2d(x, 2)
        return self.fc(x.mean((2, 3)))  # agregação global antes da classificação


class VGG16SE(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.blocos, self.ses = nn.ModuleList(), nn.ModuleList()
        c_in = 3
        for canais in [[64, 64], [128, 128], [256, 256, 256], [512, 512, 512], [512, 512, 512]]:
            camadas = []
            for c in canais:
                camadas += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU()]
                c_in = c
            self.blocos.append(nn.Sequential(*camadas))
            self.ses.append(nn.Sequential(nn.Linear(c, c // 16), nn.ReLU(), nn.Linear(c // 16, c), nn.Sigmoid()))
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        for bloco, se in zip(self.blocos, self.ses):
            x = bloco(x)
            x = x * se(x.mean((2, 3)))[:, :, None, None]  # SE: pondera cada canal
            x = nn.functional.max_pool2d(x, 2)
        return self.fc(x.mean((2, 3)))  # agregação global antes da classificação


class VGG16CBAM(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.blocos, self.mlps, self.espaciais = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        c_in = 3
        for canais in [[64, 64], [128, 128], [256, 256, 256], [512, 512, 512], [512, 512, 512]]:
            camadas = []
            for c in canais:
                camadas += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU()]
                c_in = c
            self.blocos.append(nn.Sequential(*camadas))
            self.mlps.append(nn.Sequential(nn.Linear(c, c // 16), nn.ReLU(), nn.Linear(c // 16, c)))
            self.espaciais.append(nn.Conv2d(2, 1, 7, padding=3, bias=False))
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        for bloco, mlp, espacial in zip(self.blocos, self.mlps, self.espaciais):
            x = bloco(x)
            x = x * torch.sigmoid(mlp(x.mean((2, 3))) + mlp(x.amax((2, 3))))[:, :, None, None]  # CBAM: atenção de canal
            x = x * torch.sigmoid(espacial(torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)))  # CBAM: atenção espacial
            x = nn.functional.max_pool2d(x, 2)
        return self.fc(x.mean((2, 3)))  # agregação global antes da classificação


class VGG16SECBAM(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.blocos, self.ses, self.mlps, self.espaciais = nn.ModuleList(), nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        c_in = 3
        for canais in [[64, 64], [128, 128], [256, 256, 256], [512, 512, 512], [512, 512, 512]]:
            camadas = []
            for c in canais:
                camadas += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU()]
                c_in = c
            self.blocos.append(nn.Sequential(*camadas))
            self.ses.append(nn.Sequential(nn.Linear(c, c // 16), nn.ReLU(), nn.Linear(c // 16, c), nn.Sigmoid()))
            self.mlps.append(nn.Sequential(nn.Linear(c, c // 16), nn.ReLU(), nn.Linear(c // 16, c)))
            self.espaciais.append(nn.Conv2d(2, 1, 7, padding=3, bias=False))
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        for bloco, se, mlp, espacial in zip(self.blocos, self.ses, self.mlps, self.espaciais):
            x = bloco(x)
            x = x * se(x.mean((2, 3)))[:, :, None, None]  # SE: pondera cada canal
            x = x * torch.sigmoid(mlp(x.mean((2, 3))) + mlp(x.amax((2, 3))))[:, :, None, None]  # CBAM: atenção de canal
            x = x * torch.sigmoid(espacial(torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)))  # CBAM: atenção espacial
            x = nn.functional.max_pool2d(x, 2)
        return self.fc(x.mean((2, 3)))  # agregação global antes da classificação


for Modelo in (VGG16, VGG16SE, VGG16CBAM, VGG16SECBAM):
    m = Modelo()
    print(Modelo.__name__, f"{sum(p.numel() for p in m.parameters()):,} parâmetros",
          m(torch.randn(2, 3, 128, 128)).shape, m(torch.randn(2, 3, 256, 256)).shape)

## 3. Treino das CNNs (hiperparâmetros iniciais)

Aqui as **oito CNNs** são treinadas **do zero** com os **mesmos hiperparâmetros e o mesmo laço de treino das MLPs** (`02_mlp_preliminar.ipynb`). Assim a comparação entre MLP e CNN, e entre as próprias CNNs, não depende de ajustes diferentes.

> Todos os treinos desta seção são **preliminares**. Só usam os dados de **treino** e de **validação**. O **teste não é usado**.

| Item | Valor |
|---|---|
| Otimizador | Adam |
| Velocidade de aprendizado (lr) | 0,0001 |
| Weight decay | 0,0001 |
| Imagens por lote | 64 |
| Limite de épocas | 200 |
| Parada antecipada | 10 épocas sem melhora no F1 macro da validação |
| Modelo guardado | o da época com maior F1 macro na validação |
| Perda | entropia cruzada |
| Aumento de dados (só no treino) | **leve**: espelhar e girar a imagem de 90 em 90 graus (a receita que foi melhor nas MLPs) |
| Sorteio (semente) | 42 |
| Entrada | imagem colorida `3 × R × R`, normalizada com a média e o desvio do **treino**. A imagem **não** é achatada como nas MLPs |
| Resolução `R` | `RESOLUCAO = 128` (para 256 × 256, troque o valor e rode de novo) |

**Como o notebook organiza os treinos**

- São oito treinos, um por classe de CNN, e cada um grava arquivos com o nome `{classe}-{resolução}_preliminar`, por exemplo `ResNet18SE-128_preliminar`.
- Os arquivos vão para `modelos/`, `resultados/historicos/`, `resultados/configs/`, `resultados/experimentos.csv` e `resultados/figuras/cnn/`. Nada de MLP é tocado: a função recusa nomes que começam com `MLP`.
- **Dá para interromper e continuar.** Se um treino já terminou (os arquivos dele existem), a célula de treino o pula. Para refazer tudo, use `SOBRESCREVER = True`.
- A VGG-16 faz cerca de 8 vezes mais operações que a ResNet-18, então ela é a parte demorada. A ordem do treino é: as quatro ResNets e depois as quatro VGGs.

In [ ]:
import gc
import json
import os
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from torch import nn

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLASSES = ("airport", "beach", "circular_farmland", "dense_residential", "desert",
           "forest", "harbor", "mountain", "river", "storage_tank")  # ordem fixa do projeto

RESOLUCAO = 128      # 128 ou 256
SEMENTE = 42
# Os mesmos hiperparâmetros usados nas MLPs (02_mlp_preliminar.ipynb)
LR_CNN, WD_CNN, LOTE_CNN, MAX_EPOCAS_CNN, PACIENCIA_CNN = 1e-4, 1e-4, 64, 200, 10
AUMENTO_CNN = True   # True: espelhamentos e rotações de 90° (só no treino). False: nenhum
SOBRESCREVER = False  # False: pula treinos que já terminaram. True: treina de novo


def fixar_semente():
    random.seed(SEMENTE)
    np.random.seed(SEMENTE)
    torch.manual_seed(SEMENTE)
    torch.cuda.manual_seed_all(SEMENTE)
    os.environ["PYTHONHASHSEED"] = str(SEMENTE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fixar_semente()

if torch.cuda.is_available():
    dispositivo, hardware = torch.device("cuda"), f"GPU {torch.cuda.get_device_name(0)}"
elif torch.backends.mps.is_available():
    dispositivo, hardware = torch.device("mps"), "Apple MPS"
else:
    dispositivo, hardware = torch.device("cpu"), f"CPU {platform.processor() or platform.machine()} ({os.cpu_count()} núcleos lógicos)"


def sincronizar():
    if dispositivo.type == "cuda":
        torch.cuda.synchronize()
    elif dispositivo.type == "mps":
        torch.mps.synchronize()


def liberar_memoria():
    gc.collect()
    if dispositivo.type == "cuda":
        torch.cuda.empty_cache()
    elif dispositivo.type == "mps":
        torch.mps.empty_cache()


print("torch", torch.__version__, "|", hardware, "| resolução", f"{RESOLUCAO}x{RESOLUCAO}")

### 3.1 Dados

Lê as **mesmas partições** (`dados/splits/particoes.csv`) das MLPs, redimensiona para `R × R` e guarda as imagens como inteiros de 8 bits (`uint8`), que ocupam 4 vezes menos memória. A conversão para decimal e a normalização acontecem em cada lote, já no dispositivo de treino. A média e o desvio vêm **só do treino**. O teste não é carregado.

In [ ]:
PART = pd.read_csv(RAIZ / "dados" / "splits" / "particoes.csv").sort_values("caminho_relativo")
PART["rotulo"] = PART["classe"].map({c: i for i, c in enumerate(CLASSES)})


def carregar_cnn(particao):
    sub = PART[PART["particao"] == particao]
    imagens = np.empty((len(sub), 3, RESOLUCAO, RESOLUCAO), np.uint8)
    for i, caminho in enumerate(sub["caminho_relativo"]):
        with Image.open(RAIZ / caminho) as img:
            imagens[i] = np.asarray(img.convert("RGB").resize((RESOLUCAO, RESOLUCAO), Image.BILINEAR), dtype=np.uint8).transpose(2, 0, 1)
    return torch.from_numpy(imagens), torch.from_numpy(sub["rotulo"].to_numpy(np.int64))


img_tr, y_tr = carregar_cnn("treino")
img_va, y_va = carregar_cnn("validacao")

# média e desvio por canal, só do treino (em blocos, em float64)
soma, soma2 = torch.zeros(3, dtype=torch.float64), torch.zeros(3, dtype=torch.float64)
for i in range(0, len(img_tr), 256):
    b = img_tr[i:i + 256].double() / 255
    soma += b.sum((0, 2, 3))
    soma2 += (b ** 2).sum((0, 2, 3))
n_pix = len(img_tr) * RESOLUCAO * RESOLUCAO
media = (soma / n_pix).numpy()
desvio = (soma2 / n_pix - (soma / n_pix) ** 2).sqrt().numpy()

X_tr, X_va = img_tr.to(dispositivo), img_va.to(dispositivo)
Y_tr, Y_va = y_tr.to(dispositivo), y_va.to(dispositivo)
media_t = torch.tensor(media, dtype=torch.float32, device=dispositivo).view(1, 3, 1, 1)
desvio_t = torch.tensor(desvio, dtype=torch.float32, device=dispositivo).view(1, 3, 1, 1)


def preparar(xb):
    """uint8 (N, 3, R, R) -> decimal normalizado com a média e o desvio do treino."""
    return (xb.float() / 255.0 - media_t) / desvio_t


def aumentar(x, g):
    """Aumento de dados (só no treino): espelhamentos e rotações de 90°, sorteados por imagem."""
    n = len(x)
    h = (torch.rand(n, generator=g) < 0.5).to(x.device)[:, None, None, None]
    v = (torch.rand(n, generator=g) < 0.5).to(x.device)[:, None, None, None]
    k = torch.randint(0, 4, (n,), generator=g).to(x.device)[:, None, None, None]
    x = torch.where(h, x.flip(3), x)
    x = torch.where(v, x.flip(2), x)
    for q in (1, 2, 3):
        x = torch.where(k == q, torch.rot90(x, q, (2, 3)), x)
    return x


print("Treino   :", tuple(X_tr.shape), X_tr.dtype, "|", f"{X_tr.numel() / 1024**2:.0f} MB")
print("Validação:", tuple(X_va.shape))
print("Média por canal (treino) :", np.round(media, 4))
print("Desvio por canal (treino):", np.round(desvio, 4))
print(pd.Series(y_tr.numpy()).map(dict(enumerate(CLASSES))).value_counts().reindex(CLASSES))

### 3.2 Função de treino

É o mesmo laço de treino das MLPs: Adam, entropia cruzada, parada antecipada pelo F1 macro de validação, checkpoint na melhor época e as mesmas métricas impressas (perda, acurácia, precisão, recall e F1 macro de treino e de validação, mais o relatório por classe). A função recebe os hiperparâmetros como argumentos porque a seção 4 (Optuna) usa a mesma função, de dois jeitos:

- **treino normal** (`salvar=True`): grava checkpoint, histórico, configuração e a linha em `experimentos.csv`;
- **dentro do Optuna** (`trial` informado, `salvar=False`): treina em silêncio, reporta o F1 de validação a cada época e pode ser podada.

Os pesos iniciais saem sempre da mesma semente (`fixar_semente()` roda antes de criar cada modelo), então cada treino não depende dos anteriores.

In [ ]:
DESCRICOES_AUMENTO = {False: "nenhum", True: "espelhamentos e rotações de 90° (só no treino)"}


def treinar_cnn(Modelo, arq, nome_execucao, aumento=True, lr=LR_CNN, weight_decay=WD_CNN, lote=LOTE_CNN,
                max_epocas=MAX_EPOCAS_CNN, trial=None, salvar=True, tipo="preliminar", extras=None):
    if salvar:
        assert not nome_execucao.startswith("MLP"), "nomes que começam com MLP são das MLPs e não podem ser sobrescritos"
        assert tipo != "preliminar_optuna" or "_opt" in nome_execucao, "treinos com Optuna devem ter _opt no nome"
    descricao_aumento = DESCRICOES_AUMENTO[aumento]
    extras = {"arquitetura": arq, "classe": Modelo.__name__, "resolucao": RESOLUCAO, "aumento_de_dados": descricao_aumento, **(extras or {})}
    fixar_semente()
    modelo = Modelo().to(dispositivo)
    g_aug, g_ordem = torch.Generator().manual_seed(SEMENTE), torch.Generator().manual_seed(SEMENTE)
    perda_fn = nn.CrossEntropyLoss()
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr, weight_decay=weight_decay)

    def epoca(treino):
        modelo.train(treino)
        X, Y = (X_tr, Y_tr) if treino else (X_va, Y_va)
        n = len(Y)
        ordem = torch.randperm(n, generator=g_ordem).to(dispositivo) if treino else torch.arange(n, device=dispositivo)
        tamanho = lote if treino else 256
        soma, previstas, reais = 0.0, [], []
        with torch.set_grad_enabled(treino):
            for i in range(0, n, tamanho):
                idx = ordem[i:i + tamanho]
                if treino and len(idx) < 2:      # BatchNorm não aceita lote de 1 imagem
                    continue
                xb, yb = preparar(X[idx]), Y[idx]
                if treino and aumento:
                    xb = aumentar(xb, g_aug)
                logits = modelo(xb)
                perda = perda_fn(logits, yb)
                if treino:
                    otimizador.zero_grad(set_to_none=True)
                    perda.backward()
                    otimizador.step()
                soma += perda.item() * len(yb)
                previstas.append(logits.argmax(1).cpu())
                reais.append(yb.cpu())
        p, r = torch.cat(previstas).numpy(), torch.cat(reais).numpy()
        macro = dict(average="macro", zero_division=0)  # classe sem previsões: precisão = 0 (sem divisão por zero)
        return (soma / len(r), accuracy_score(r, p), precision_score(r, p, **macro),
                recall_score(r, p, **macro), f1_score(r, p, **macro))

    caminho_ckpt = RAIZ / "modelos" / f"{nome_execucao}.pt"
    if salvar:
        (RAIZ / "modelos").mkdir(exist_ok=True)
    historico, melhor_f1, melhor_epoca, sem_melhora, tempo_total = [], -1.0, 0, 0, 0.0

    for n in range(1, max_epocas + 1):
        sincronizar()
        t0 = time.perf_counter()
        tr = epoca(True)
        va = epoca(False)
        sincronizar()
        dt = time.perf_counter() - t0
        tempo_total += dt
        historico.append({"epoca": n,
                          "perda_treino": tr[0], "acuracia_treino": tr[1], "precisao_macro_treino": tr[2],
                          "recall_macro_treino": tr[3], "f1_macro_treino": tr[4],
                          "perda_validacao": va[0], "acuracia_validacao": va[1], "precisao_macro_validacao": va[2],
                          "recall_macro_validacao": va[3], "f1_macro_validacao": va[4], "tempo_epoca_s": dt})
        melhorou = va[4] > melhor_f1
        if melhorou:
            melhor_f1, melhor_epoca, sem_melhora = va[4], n, 0
            if salvar:
                torch.save({"state_dict": modelo.state_dict(), "epoca": n, "f1_macro_validacao": va[4]}, caminho_ckpt)
        else:
            sem_melhora += 1
        if salvar:
            print(f"[{nome_execucao}] época {n:3d}/{max_epocas} | "
                  f"treino: perda {tr[0]:.4f} acc {tr[1]:.4f} P {tr[2]:.4f} R {tr[3]:.4f} F1 {tr[4]:.4f} | "
                  f"val: perda {va[0]:.4f} acc {va[1]:.4f} P {va[2]:.4f} R {va[3]:.4f} F1 {va[4]:.4f} | {dt:5.1f}s"
                  + ("  *checkpoint*" if melhorou else ""))
        if trial is not None:                      # Optuna: reporta e deixa o Optuna podar a tentativa
            trial.report(va[4], n)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if sem_melhora >= PACIENCIA_CNN:
            if salvar:
                print(f"Parada antecipada: {PACIENCIA_CNN} épocas sem melhora no F1 macro de validação.")
            break

    hist = pd.DataFrame(historico)
    hist["checkpoint"] = hist["epoca"] == melhor_epoca
    if not salvar:
        return None, hist

    (RAIZ / "resultados" / "historicos").mkdir(parents=True, exist_ok=True)
    hist.to_csv(RAIZ / "resultados" / "historicos" / f"{nome_execucao}_historico.csv", index=False)
    ck = hist[hist["checkpoint"]].iloc[0]
    resumo = {
        "nome_execucao": nome_execucao, "tipo_execucao": tipo,
        "parametros": sum(p.numel() for p in modelo.parameters()),
        "tamanho_modelo_mb": round(caminho_ckpt.stat().st_size / 1024**2, 2),
        "epocas_executadas": len(hist), "epoca_checkpoint": melhor_epoca,
        "f1_macro_validacao": round(float(ck["f1_macro_validacao"]), 4),
        "acuracia_validacao": round(float(ck["acuracia_validacao"]), 4),
        "perda_validacao": round(float(ck["perda_validacao"]), 4),
        "tempo_total_s": round(tempo_total, 1), "tempo_medio_epoca_s": round(tempo_total / len(hist), 2),
        "hardware": hardware, "otimizador": "adam", "taxa_aprendizagem": lr, "weight_decay": weight_decay,
        "tamanho_lote": lote, "max_epocas": max_epocas, "paciencia": PACIENCIA_CNN,
        "criterio_parada": "F1 macro de validação", "semente": SEMENTE, "torch": torch.__version__,
        "extras": json.dumps(extras, ensure_ascii=False),
    }
    print()
    print(f"Resumo {nome_execucao}: checkpoint na época {melhor_epoca} de {len(hist)} | {resumo['parametros']:,} parâmetros | "
          f"{tempo_total:.1f}s no total ({tempo_total / len(hist):.2f}s/época)")
    for rotulo, sufixo in (("treino   ", "treino"), ("validação", "validacao")):
        print(f"  {rotulo} : perda {ck['perda_' + sufixo]:.4f} | acurácia {ck['acuracia_' + sufixo]:.4f} | "
              f"precisão macro {ck['precisao_macro_' + sufixo]:.4f} | recall macro {ck['recall_macro_' + sufixo]:.4f} | "
              f"F1 macro {ck['f1_macro_' + sufixo]:.4f}")
    print(f"  diferença treino − validação (F1 macro): {ck['f1_macro_treino'] - ck['f1_macro_validacao']:+.4f}")
    modelo.load_state_dict(torch.load(caminho_ckpt, map_location=dispositivo)["state_dict"])
    modelo.eval()
    with torch.no_grad():
        previstas = torch.cat([modelo(preparar(X_va[i:i + 256])).argmax(1).cpu() for i in range(0, len(Y_va), 256)]).numpy()
    print()
    print(f"Relatório por classe (validação, checkpoint da época {melhor_epoca}):")
    print(classification_report(y_va.numpy(), previstas, target_names=CLASSES, digits=4, zero_division=0))

    csv_exp = RAIZ / "resultados" / "experimentos.csv"
    novo = pd.DataFrame([resumo])
    if csv_exp.is_file():
        antigo = pd.read_csv(csv_exp)
        novo = pd.concat([antigo[antigo["nome_execucao"] != nome_execucao], novo], ignore_index=True)
    novo.to_csv(csv_exp, index=False)
    (RAIZ / "resultados" / "configs").mkdir(parents=True, exist_ok=True)
    with open(RAIZ / "resultados" / "configs" / f"{nome_execucao}.json", "w", encoding="utf-8") as f:
        json.dump({"arquitetura": arq, "entrada": f"RGB {RESOLUCAO}x{RESOLUCAO} (3x{RESOLUCAO}x{RESOLUCAO}), sem achatar",
                   "classes": list(CLASSES), "normalizacao": {"media": media.tolist(), "desvio": desvio.tolist()},
                   "aumento_de_dados": descricao_aumento, "perda": "CrossEntropyLoss", "resultado": resumo},
                  f, indent=2, ensure_ascii=False)
    return resumo, hist


def resultado_salvo(nome_execucao):
    """Se o treino já terminou (modelo, histórico e linha em experimentos.csv existem), devolve (resumo, histórico)."""
    hist_csv = RAIZ / "resultados" / "historicos" / f"{nome_execucao}_historico.csv"
    exp_csv = RAIZ / "resultados" / "experimentos.csv"
    if hist_csv.is_file() and exp_csv.is_file() and (RAIZ / "modelos" / f"{nome_execucao}.pt").is_file():
        linha = pd.read_csv(exp_csv)
        linha = linha[linha["nome_execucao"] == nome_execucao]
        if len(linha):
            return linha.iloc[0].to_dict(), pd.read_csv(hist_csv)
    return None


def plotar_curvas_cnn(hist, nome_execucao):
    ep = int(hist.loc[hist["checkpoint"], "epoca"].iloc[0])
    fig, eixos = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, (chave, titulo) in zip(eixos, [("perda", "Perda (entropia cruzada)"), ("acuracia", "Acurácia"), ("f1_macro", "F1 macro")]):
        ax.plot(hist["epoca"], hist[f"{chave}_treino"], color="#2a78b5", lw=2, label="Treino")
        ax.plot(hist["epoca"], hist[f"{chave}_validacao"], color="#d9762b", lw=2, label="Validação")
        ax.axvline(ep, color="#555", ls="--", lw=1)
        ax.scatter([ep], [hist.loc[hist["epoca"] == ep, f"{chave}_validacao"].iloc[0]], color="#d9762b", zorder=3, s=40)
        ax.set_title(titulo); ax.set_xlabel("Época"); ax.grid(alpha=0.25)
    eixos[0].legend(frameon=False)
    fig.suptitle(f"{nome_execucao} — curvas de aprendizagem (checkpoint na época {ep})", fontweight="bold")
    fig.text(0.01, 0.01, "Fonte: elaborado pelos autores (NWPU-RESISC45, 10 classes).", fontsize=8, color="#666")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    (RAIZ / "resultados" / "figuras" / "cnn").mkdir(parents=True, exist_ok=True)
    fig.savefig(RAIZ / "resultados" / "figuras" / "cnn" / f"{nome_execucao}_curvas.png", dpi=150)
    plt.show()

### 3.3 Treinando as oito CNNs

Cada CNN é treinada com `lr 1e-4`, `weight decay 1e-4`, lote 64, até 200 épocas (parada antecipada com paciência 10) e aumento leve. A linha impressa a cada época mostra perda, acurácia, precisão, recall e F1 macro de treino e de validação. No fim de cada treino saem o resumo, o relatório por classe e as curvas.

Se o notebook for interrompido, basta rodar a célula de novo: ela pula as CNNs que já terminaram e continua da próxima.

In [ ]:
MODELOS = {"ResNet18": ResNet18, "ResNet18SE": ResNet18SE, "ResNet18CBAM": ResNet18CBAM, "ResNet18SECBAM": ResNet18SECBAM,
           "VGG16": VGG16, "VGG16SE": VGG16SE, "VGG16CBAM": VGG16CBAM, "VGG16SECBAM": VGG16SECBAM}
FAMILIA = {arq: ("ResNet18" if arq.startswith("ResNet") else "VGG16") for arq in MODELOS}

resultados_cnn = {}
for arq, Modelo in MODELOS.items():
    nome_exec = f"{arq}-{RESOLUCAO}_preliminar"
    salvo = None if SOBRESCREVER else resultado_salvo(nome_exec)
    if salvo is not None:
        print(f"##### {nome_exec}: já treinado (F1 val {salvo[0]['f1_macro_validacao']:.4f}), pulando. Use SOBRESCREVER = True para refazer.")
        resultados_cnn[arq] = salvo
        continue
    print(f"##### {nome_exec} | lr {LR_CNN:.0e}  weight_decay {WD_CNN:.0e}  lote {LOTE_CNN}  aumento {'leve' if AUMENTO_CNN else 'nenhum'}")
    resultados_cnn[arq] = treinar_cnn(Modelo, arq, nome_exec, aumento=AUMENTO_CNN, tipo="preliminar",
                                      extras={"familia": FAMILIA[arq], "busca": "hiperparâmetros iniciais"})
    plotar_curvas_cnn(resultados_cnn[arq][1], nome_exec)
    liberar_memoria()

### 3.4 Comparação das oito CNNs

`dif_f1_treino_val` é o F1 macro de treino menos o de validação no melhor ponto (quanto maior, mais a rede decorou o treino). `s/época` é o tempo médio de uma época nesta máquina.

In [ ]:
VARIANTES = [("", "sem atenção"), ("SE", "SE"), ("CBAM", "CBAM"), ("SECBAM", "SE + CBAM")]
COR_FAMILIA = {"ResNet18": "#2a78d6", "VGG16": "#eb6834"}


def tabela_cnn(sufixo):
    exp = pd.read_csv(RAIZ / "resultados" / "experimentos.csv").set_index("nome_execucao")
    linhas = []
    for arq in MODELOS:
        n = f"{arq}-{RESOLUCAO}{sufixo}"
        if n not in exp.index:
            continue
        h = pd.read_csv(RAIZ / "resultados" / "historicos" / f"{n}_historico.csv")
        ck = h[h["checkpoint"]].iloc[0]
        linhas.append({"CNN": arq, "parâmetros": int(exp.loc[n, "parametros"]), "F1 val": exp.loc[n, "f1_macro_validacao"],
                       "acurácia val": exp.loc[n, "acuracia_validacao"],
                       "dif_f1_treino_val": round(ck["f1_macro_treino"] - ck["f1_macro_validacao"], 3),
                       "época ckpt": int(exp.loc[n, "epoca_checkpoint"]), "épocas": int(exp.loc[n, "epocas_executadas"]),
                       "s/época": exp.loc[n, "tempo_medio_epoca_s"], "tempo total (s)": exp.loc[n, "tempo_total_s"]})
    return pd.DataFrame(linhas).set_index("CNN")


comparacao_cnn = tabela_cnn("_preliminar")
print(comparacao_cnn.to_string())
comparacao_cnn

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

ESTILO = {"SUP": "#fcfcfb", "TINTA": "#0b0b0b", "TINTA_2": "#52514e", "TINTA_3": "#8a8984", "GRADE": "#e6e5e1"}
FONTE_CNN = "Fonte: elaborado pelos autores (NWPU-RESISC45, 10 classes). Dados de validação; o teste não foi usado."


def estilo_graficos():
    e = ESTILO
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "text.color": e["TINTA"], "axes.labelcolor": e["TINTA_2"],
                         "xtick.color": e["TINTA_2"], "ytick.color": e["TINTA_2"], "axes.edgecolor": e["GRADE"],
                         "axes.facecolor": e["SUP"], "figure.facecolor": e["SUP"], "savefig.facecolor": e["SUP"],
                         "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": e["GRADE"],
                         "grid.linewidth": 0.8, "axes.axisbelow": True, "legend.frameon": False})


def cabecalho(fig, titulo, sub):
    fig.text(0.055, 0.965, titulo, fontsize=14, fontweight="bold", ha="left", va="top")
    fig.text(0.055, 0.915, sub, fontsize=10, color=ESTILO["TINTA_2"], ha="left", va="top")
    fig.text(0.055, 0.015, FONTE_CNN, fontsize=8, color=ESTILO["TINTA_3"], ha="left", va="bottom")


def legenda_familias(fig, y=0.865):
    hs = [Line2D([0], [0], marker="s", color="none", markerfacecolor=c, markeredgecolor="none", markersize=10, label=f) for f, c in COR_FAMILIA.items()]
    fig.legend(handles=hs, loc="upper left", bbox_to_anchor=(0.055, y), ncol=2, fontsize=10, labelcolor=ESTILO["TINTA_2"], handletextpad=0.3)


def graficos_cnn(sufixo, titulo, arquivo_f1, arquivo_custo):
    estilo_graficos()
    tab = tabela_cnn(sufixo)
    pt = lambda x, d=3: f"{x:.{d}f}".replace(".", ",")
    virgula = FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ","))
    saida = RAIZ / "resultados" / "figuras" / "cnn"
    saida.mkdir(parents=True, exist_ok=True)

    # 1) F1 macro de validação: um grupo por tipo de atenção, uma barra por família
    fig, ax = plt.subplots(figsize=(10, 5.8))
    fig.subplots_adjust(left=0.08, right=0.97, top=0.78, bottom=0.12)
    larg = 0.36
    for i, (var, rot) in enumerate(VARIANTES):
        for j, fam in enumerate(COR_FAMILIA):
            arq = f"{fam}{var}"
            if arq not in tab.index:
                continue
            x, v = i + (j - 0.5) * (larg + 0.02), tab.loc[arq, "F1 val"]
            ax.bar(x, v, width=larg, color=COR_FAMILIA[fam], edgecolor=ESTILO["SUP"], linewidth=2, zorder=3)
            ax.text(x, v + 0.012, pt(v), ha="center", va="bottom", fontsize=10.5, fontweight="bold")
    ax.set_xticks(range(len(VARIANTES))); ax.set_xticklabels([r for _, r in VARIANTES], fontsize=11, color=ESTILO["TINTA"])
    ax.set_ylim(0, 1.0); ax.set_ylabel("F1 macro na validação"); ax.yaxis.set_major_formatter(virgula)
    ax.grid(axis="x", visible=False); ax.tick_params(axis="x", length=0)
    cabecalho(fig, titulo, f"F1 macro na validação ({RESOLUCAO} × {RESOLUCAO}), no melhor ponto de cada treino (maior é melhor)")
    legenda_familias(fig)
    fig.savefig(saida / arquivo_f1, dpi=200); plt.show()

    # 2) custo × desempenho: tempo médio por época × F1 macro (ResNet com o rótulo acima do ponto, VGG abaixo)
    fig, ax = plt.subplots(figsize=(10, 5.8))
    fig.subplots_adjust(left=0.09, right=0.96, top=0.78, bottom=0.15)
    for arq, linha in tab.iterrows():
        fam = FAMILIA[arq]
        ax.plot([linha["s/época"]], [linha["F1 val"]], marker="o", markersize=13, color=COR_FAMILIA[fam],
                markeredgecolor=ESTILO["SUP"], markeredgewidth=2, linestyle="none", zorder=4)
        rot = dict(VARIANTES)[arq[len(fam):]]
        ax.annotate(rot, (linha["s/época"], linha["F1 val"]), xytext=(11, 0), textcoords="offset points",
                    ha="left", va="center", fontsize=9.5, color=ESTILO["TINTA_2"])
    ax.set_xlabel("Tempo médio por época (segundos)"); ax.set_ylabel("F1 macro na validação"); ax.yaxis.set_major_formatter(virgula)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",")))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.2f}".replace(".", ",")))  # faixa estreita: 2 casas
    lo, hi = tab["F1 val"].min(), tab["F1 val"].max()
    ax.set_ylim(max(0, np.floor((lo - 0.05) * 20) / 20), min(1.0, np.ceil((hi + 0.05) * 20) / 20))
    ax.set_xlim(0, tab["s/época"].max() * 1.3); ax.grid(axis="x", visible=False)
    cabecalho(fig, "Custo × desempenho das CNNs", f"Tempo por época × F1 macro na validação ({RESOLUCAO} × {RESOLUCAO}; tempo medido em {hardware})")
    legenda_familias(fig)
    fig.savefig(saida / arquivo_custo, dpi=200); plt.show()


graficos_cnn("_preliminar", "Qual CNN foi melhor, com os hiperparâmetros iniciais",
             f"cnn_comparacao_1_f1_validacao_{RESOLUCAO}.png", f"cnn_comparacao_2_custo_desempenho_{RESOLUCAO}.png")

---

## 4. Hiperparâmetros das CNNs com Optuna

As seções 1 a 3 **não são alteradas** por esta parte. Aqui o [Optuna](https://optuna.org) busca hiperparâmetros melhores para as CNNs e depois as oito CNNs são **treinadas de novo** com eles, **sem sobrescrever** nada da seção 3: os arquivos novos levam `_opt` no nome (`ResNet18SE-128_opt_preliminar`).

**Regras que seguimos (enunciado do projeto)**

- A busca é uma **busca adicional**, identificada à parte das execuções finais.
- Quem decide é o **F1 macro na validação**. O **teste não é usado**.
- A semente é a mesma (42) em todas as tentativas.

**Como a busca foi montada**

| Item | Valor |
|---|---|
| Quantas buscas | **uma por família**: uma para a ResNet-18 e outra para a VGG-16, feitas na versão **sem atenção** (`ResNet18` e `VGG16`) |
| Por que por família | treinar uma busca para cada uma das oito CNNs custaria quatro vezes mais. Com os mesmos hiperparâmetros nas quatro variantes de uma família, a **única diferença entre elas continua sendo o módulo de atenção** |
| Aumento de dados nas tentativas | leve (espelhamentos e rotações de 90°), o mesmo do treino da seção 3 |
| O que se maximiza | o melhor F1 macro de validação da tentativa |
| Tentativas por família | `N_TRIALS_CNN` (padrão 10) |
| Limite de épocas por tentativa | `EPOCAS_TRIAL_CNN` (padrão 30), com a mesma parada antecipada (paciência 10) |
| Poda | `MedianPruner`: corta a tentativa que fica abaixo da mediana das anteriores |
| Resolução da busca | a de `RESOLUCAO` (128). Os hiperparâmetros achados também serão usados em 256 × 256, para a resolução ser a única diferença |

**Hiperparâmetros buscados**

| Hiperparâmetro | Faixa | O que é |
|---|---|---|
| `lr` | 1e-5 a 3e-3 (escala log) | velocidade de aprendizado |
| `weight_decay` | 1e-6 a 1e-2 (escala log) | multa para pesos grandes |
| `lote` | 32, 64 ou 128 | imagens por lote |

As arquiteturas **não mudam**, e as CNNs não têm dropout. O estudo fica salvo em `resultados/optuna/cnn.db`: se o notebook for interrompido, basta rodar a célula de novo e ela continua de onde parou. Para recomeçar do zero, apague esse arquivo.

**Tempo:** cada tentativa treina uma CNN inteira (até 30 épocas), então esta é a parte mais demorada do projeto, sobretudo na VGG-16. Reduza `N_TRIALS_CNN` ou `EPOCAS_TRIAL_CNN` se precisar.

In [ ]:
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS_CNN = 10         # tentativas por família
EPOCAS_TRIAL_CNN = 30     # limite de épocas em cada tentativa (a parada antecipada continua valendo)
ESTUDOS_CNN = {"ResNet18": ResNet18, "VGG16": VGG16}   # uma busca por família, na versão sem atenção
DIR_OPTUNA = RAIZ / "resultados" / "optuna"
DIR_OPTUNA.mkdir(parents=True, exist_ok=True)
print("optuna", optuna.__version__, "| tentativas por família:", N_TRIALS_CNN, "| épocas por tentativa:", EPOCAS_TRIAL_CNN)

### 4.1 Rodando os estudos

Cada tentativa sorteia `lr`, `weight_decay` e `lote`, treina a CNN com aumento leve e devolve o melhor F1 macro de validação. A linha impressa mostra a tentativa, se ela foi concluída ou podada, o F1 dela e o melhor até agora.

In [ ]:
def criar_objetivo(Modelo):
    def objetivo(trial):
        hp = dict(lr=trial.suggest_float("lr", 1e-5, 3e-3, log=True),
                  weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
                  lote=trial.suggest_categorical("lote", [32, 64, 128]))
        _, hist = treinar_cnn(Modelo, Modelo.__name__, f"{Modelo.__name__}-{RESOLUCAO}_trial{trial.number}", aumento=AUMENTO_CNN,
                              lr=hp["lr"], weight_decay=hp["weight_decay"], lote=hp["lote"], max_epocas=EPOCAS_TRIAL_CNN,
                              trial=trial, salvar=False)
        liberar_memoria()
        return float(hist["f1_macro_validacao"].max())
    return objetivo


def mostrar_tentativa(familia):
    def callback(estudo, t):
        valor = f"{t.value:.4f}" if t.value is not None else "  -   "
        melhor = f"{estudo.best_value:.4f}" if any(x.state.name == "COMPLETE" for x in estudo.trials) else "  -   "
        print(f"[{familia}] tentativa {t.number:2d} | {t.state.name:<8} | F1 val {valor} | melhor {melhor} | "
              f"lr {t.params['lr']:.2e}  wd {t.params['weight_decay']:.2e}  lote {t.params['lote']}")
    return callback


estudos_cnn = {}
for familia, Modelo in ESTUDOS_CNN.items():
    estudo = optuna.create_study(
        study_name=f"{familia}-{RESOLUCAO}", direction="maximize", sampler=TPESampler(seed=SEMENTE),
        pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=5),
        storage=f"sqlite:///{(DIR_OPTUNA / 'cnn.db').as_posix()}", load_if_exists=True)
    faltam = N_TRIALS_CNN - len(estudo.trials)
    print(f"=== Estudo {familia}-{RESOLUCAO}: {len(estudo.trials)} tentativas já feitas, rodando mais {max(faltam, 0)} ===")
    if faltam > 0:
        estudo.optimize(criar_objetivo(Modelo), n_trials=faltam, callbacks=[mostrar_tentativa(familia)])
    estudos_cnn[familia] = estudo

### 4.2 Melhores hiperparâmetros

In [ ]:
linhas = []
for familia, estudo in estudos_cnn.items():
    estados = pd.Series([t.state.name for t in estudo.trials]).value_counts().to_dict()
    linhas.append({"família": f"{familia}-{RESOLUCAO}", "tentativas": len(estudo.trials), "concluídas": estados.get("COMPLETE", 0),
                   "podadas": estados.get("PRUNED", 0), "melhor F1 val": round(estudo.best_value, 4),
                   "lr": estudo.best_params["lr"], "weight_decay": estudo.best_params["weight_decay"], "lote": estudo.best_params["lote"]})
    estudo.trials_dataframe().to_csv(DIR_OPTUNA / f"tentativas_{familia}-{RESOLUCAO}.csv", index=False)

tabela_hp_cnn = pd.DataFrame(linhas).set_index("família")
melhores_cnn = {f: dict(e.best_params) for f, e in estudos_cnn.items()}
with open(DIR_OPTUNA / "melhores_hiperparametros_cnn.json", "w", encoding="utf-8") as f:
    json.dump({f"{fam}-{RESOLUCAO}": {**hp, "melhor_f1_macro_validacao": estudos_cnn[fam].best_value} for fam, hp in melhores_cnn.items()},
              f, indent=2, ensure_ascii=False)
print("Valores salvos em resultados/optuna/ (tentativas_*.csv e melhores_hiperparametros_cnn.json)")
tabela_hp_cnn

### 4.3 Gráficos da busca

**Histórico:** cada ponto é uma tentativa (os cinzas foram podadas), e a linha é o melhor F1 até aquele momento. **Importância:** quanto cada hiperparâmetro influenciou o resultado, segundo o Optuna.

In [ ]:
def graficos_optuna_cnn(estudos, saida):
    estilo_graficos()
    e = ESTILO
    AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#b9b8b3"
    virgula2 = FuncFormatter(lambda v, _: f"{v:.2f}".replace(".", ","))
    virgula1 = FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ","))
    saida.mkdir(parents=True, exist_ok=True)

    fig, eixos = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
    for ax, (familia, estudo) in zip(eixos, estudos.items()):
        ts = [t for t in estudo.trials if t.state.name in ("COMPLETE", "PRUNED")]
        for t in ts:
            ok = t.state.name == "COMPLETE"
            v = t.value if ok else max((x for x in t.intermediate_values.values()), default=None)
            if v is not None:
                ax.plot([t.number], [v], "o", color=AZUL if ok else CINZA, markersize=8, markeredgecolor=e["SUP"], markeredgewidth=1.5, zorder=3)
        melhor, acumulado = -1, []
        for t in sorted(ts, key=lambda x: x.number):
            if t.state.name == "COMPLETE":
                melhor = max(melhor, t.value)
            acumulado.append((t.number, melhor if melhor > 0 else None))
        ax.plot([n for n, v in acumulado if v], [v for n, v in acumulado if v], color=LARANJA, lw=2.2, zorder=2)
        ax.set_title(f"{familia}-{RESOLUCAO}", loc="left", fontsize=11, color=e["TINTA"])
        ax.set_xlabel("Tentativa"); ax.yaxis.set_major_formatter(virgula2)
    eixos[0].set_ylabel("F1 macro na validação")
    fig.text(0.055, 0.97, "Histórico da busca com Optuna (CNNs)", fontsize=14, fontweight="bold", ha="left", va="top")
    fig.text(0.055, 0.915, "Azul: tentativa concluída. Cinza: podada. Linha laranja: melhor F1 até a tentativa.", fontsize=10, color=e["TINTA_2"], ha="left", va="top")
    fig.text(0.055, 0.01, FONTE_CNN, fontsize=8, color=e["TINTA_3"])
    fig.subplots_adjust(left=0.07, right=0.98, top=0.8, bottom=0.14, wspace=0.08)
    fig.savefig(saida / f"optuna_cnn_historico_{RESOLUCAO}.png", dpi=200); plt.show()

    fig, eixos = plt.subplots(1, 2, figsize=(12, 3.8))
    algum = False
    for ax, (familia, estudo) in zip(eixos, estudos.items()):
        try:
            imp = optuna.importance.get_param_importances(estudo)
        except Exception:
            ax.set_title(f"{familia}-{RESOLUCAO}: poucas tentativas para estimar", loc="left", fontsize=10, color=e["TINTA_2"]); ax.axis("off"); continue
        algum = True
        nomes, valores = list(imp.keys())[::-1], list(imp.values())[::-1]
        ax.barh(nomes, valores, color=AZUL, edgecolor=e["SUP"], linewidth=2, height=0.6, zorder=3)
        for y, v in enumerate(valores):
            ax.text(v + 0.01, y, f"{v:.2f}".replace(".", ","), va="center", fontsize=10, color=e["TINTA"])
        ax.set_xlim(0, max(valores) * 1.2); ax.set_title(f"{familia}-{RESOLUCAO}", loc="left", fontsize=11, color=e["TINTA"])
        ax.set_xlabel("Importância relativa"); ax.grid(axis="y", visible=False); ax.xaxis.set_major_formatter(virgula1)
    fig.text(0.055, 0.97, "O que mais pesou no resultado (CNNs)", fontsize=14, fontweight="bold", ha="left", va="top")
    fig.subplots_adjust(left=0.12, right=0.97, top=0.78, bottom=0.17, wspace=0.35)
    if algum:
        fig.savefig(saida / f"optuna_cnn_importancia_{RESOLUCAO}.png", dpi=200)
    plt.show()


graficos_optuna_cnn(estudos_cnn, RAIZ / "resultados" / "figuras" / "cnn")

### 4.4 Retreino das oito CNNs com os melhores hiperparâmetros

Cada CNN usa os melhores hiperparâmetros da **sua família** (`lr`, `weight_decay` e `lote`). O resto é igual à seção 3: aumento leve, limite de 200 épocas, paciência 10 e semente 42. Assim a diferença para a seção 3 é só o ajuste do Optuna.

- Os nomes terminam em `_opt_preliminar` (por exemplo `ResNet18CBAM-128_opt_preliminar`). **Os treinos da seção 3 não são tocados.**
- Os arquivos novos entram em `modelos/`, `resultados/historicos/`, `resultados/configs/`, `resultados/experimentos.csv` (com `tipo_execucao = preliminar_optuna`) e `resultados/figuras/cnn/`.
- Dá para interromper e continuar: a célula pula as CNNs que já terminaram (use `SOBRESCREVER = True` para refazer).

In [ ]:
# se o kernel foi reiniciado depois da busca, os melhores valores são lidos do arquivo salvo
if "melhores_cnn" not in globals():
    with open(DIR_OPTUNA / "melhores_hiperparametros_cnn.json", encoding="utf-8") as f:
        salvos = json.load(f)
    melhores_cnn = {fam: salvos[f"{fam}-{RESOLUCAO}"] for fam in ESTUDOS_CNN}

resultados_cnn_opt = {}
for arq, Modelo in MODELOS.items():
    hp = melhores_cnn[FAMILIA[arq]]
    nome_exec = f"{arq}-{RESOLUCAO}_opt_preliminar"
    salvo = None if SOBRESCREVER else resultado_salvo(nome_exec)
    if salvo is not None:
        print(f"##### {nome_exec}: já treinado (F1 val {salvo[0]['f1_macro_validacao']:.4f}), pulando. Use SOBRESCREVER = True para refazer.")
        resultados_cnn_opt[arq] = salvo
        continue
    print(f"##### {nome_exec} | lr {hp['lr']:.2e}  weight_decay {hp['weight_decay']:.2e}  lote {hp['lote']}  (estudo {FAMILIA[arq]}-{RESOLUCAO})")
    resultados_cnn_opt[arq] = treinar_cnn(Modelo, arq, nome_exec, aumento=AUMENTO_CNN, lr=hp["lr"], weight_decay=hp["weight_decay"],
                                          lote=hp["lote"], max_epocas=MAX_EPOCAS_CNN, tipo="preliminar_optuna",
                                          extras={"familia": FAMILIA[arq], "busca": "optuna", "estudo": f"{FAMILIA[arq]}-{RESOLUCAO}"})
    plotar_curvas_cnn(resultados_cnn_opt[arq][1], nome_exec)
    liberar_memoria()

## 5. Antes × depois do Optuna (CNNs)

Compara cada CNN nos treinos da seção 3 (hiperparâmetros iniciais: lr 1e-4, weight decay 1e-4 e lote 64) com os novos (hiperparâmetros do Optuna). `dif_f1_treino_val` é o F1 de treino menos o de validação no melhor ponto.

In [ ]:
comparacao_cnn_opt = tabela_cnn("_opt_preliminar")
print(comparacao_cnn_opt.to_string())

antes_depois = pd.DataFrame({
    "F1 val antes": comparacao_cnn["F1 val"], "F1 val Optuna": comparacao_cnn_opt["F1 val"],
    "ganho": (comparacao_cnn_opt["F1 val"] - comparacao_cnn["F1 val"]).round(4),
    "dif_f1_treino_val antes": comparacao_cnn["dif_f1_treino_val"], "dif_f1_treino_val Optuna": comparacao_cnn_opt["dif_f1_treino_val"],
    "épocas antes": comparacao_cnn["épocas"], "épocas Optuna": comparacao_cnn_opt["épocas"],
    "tempo antes (s)": comparacao_cnn["tempo total (s)"], "tempo Optuna (s)": comparacao_cnn_opt["tempo total (s)"]})
antes_depois

In [ ]:
graficos_cnn("_opt_preliminar", "Qual CNN foi melhor, com os hiperparâmetros do Optuna",
             f"cnn_optuna_1_f1_validacao_{RESOLUCAO}.png", f"cnn_optuna_2_custo_desempenho_{RESOLUCAO}.png")


def grafico_antes_depois_cnn(tab, saida):
    estilo_graficos()
    ANTES = "#cfcec9"
    virgula1 = FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ","))
    fig, ax = plt.subplots(figsize=(13, 5.8))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.82, bottom=0.2)
    larg = 0.36
    for i, (arq, linha) in enumerate(tab.iterrows()):
        a, d = linha["F1 val antes"], linha["F1 val Optuna"]
        ax.bar(i - larg / 2 - 0.01, a, width=larg, color=ANTES, edgecolor=ESTILO["SUP"], linewidth=2, zorder=3)
        ax.bar(i + larg / 2 + 0.01, d, width=larg, color=COR_FAMILIA[FAMILIA[arq]], edgecolor=ESTILO["SUP"], linewidth=2, zorder=3)
        ax.text(i - larg / 2 - 0.01, a + 0.01, f"{a:.3f}".replace(".", ","), ha="center", fontsize=8.5, color=ESTILO["TINTA_2"])
        ax.text(i + larg / 2 + 0.01, d + 0.01, f"{d:.3f}".replace(".", ","), ha="center", fontsize=9, fontweight="bold")
        g = linha["ganho"]
        ax.text(i, max(a, d) + 0.055, f"{'+' if g >= 0 else '−'}{abs(g):.3f}".replace(".", ","), ha="center", fontsize=10, fontweight="bold")
    ax.set_xticks(range(len(tab))); ax.set_xticklabels(list(tab.index), fontsize=9, color=ESTILO["TINTA"], rotation=20)
    ax.set_ylim(0, 1.05); ax.set_ylabel("F1 macro na validação"); ax.yaxis.set_major_formatter(virgula1)
    ax.grid(axis="x", visible=False); ax.tick_params(axis="x", length=0)
    cabecalho(fig, "Efeito do Optuna sobre cada CNN", "Cinza: hiperparâmetros iniciais. Colorido: hiperparâmetros do Optuna (azul ResNet-18, laranja VGG-16). O número acima é o ganho.")
    saida.mkdir(parents=True, exist_ok=True)
    fig.savefig(saida / f"cnn_optuna_3_antes_depois_{RESOLUCAO}.png", dpi=200); plt.show()


grafico_antes_depois_cnn(antes_depois, RAIZ / "resultados" / "figuras" / "cnn")

## 6. Leitura dos resultados das CNNs (preencher após rodar)

- Qual CNN teve o maior F1 macro de validação com os hiperparâmetros iniciais? E com o Optuna? …
- Os módulos de atenção (SE, CBAM, SE + CBAM) ajudaram em cada família? Quanto, e a que custo de tempo? …
- ResNet-18 ou VGG-16: qual rendeu mais para o tempo gasto? …
- Melhores hiperparâmetros de cada família e como diferem dos iniciais: …
- O Optuna melhorou o F1 de validação? A diferença entre treino e validação mudou? …
- Custo da busca: número de tentativas, podadas e tempo total: …
- Cuidado: a busca foi feita na versão sem atenção e em 128 × 128. As variantes com atenção e a resolução 256 × 256 usam os mesmos valores, que podem não ser os ideais para elas. …

## 7. Bônus: YOLO em modo *classify*

Ajuste fino de um **YOLO de classificação** (Ultralytics) com pesos pré-treinados, nas dez classes e **nas mesmas partições** (`dados/splits/particoes.csv`) usadas pelas MLPs e CNNs. Esse modo classifica a imagem inteira e dispensa caixas delimitadoras. O modelo será comparado depois com a CNN obrigatória de maior F1 macro de validação, no mesmo teste.

| Item | Valor |
|---|---|
| Modelo e pesos | `yolo26n-cls.pt`, pré-treinado no ImageNet (baixado pelo Ultralytics) |
| Biblioteca | `ultralytics` 8.4.216 |
| Resolução | 256 × 256 (resolução original das imagens) |
| Épocas, lote, paciência | 50, 32 e 10 |
| Semente | 42 |
| Demais hiperparâmetros e transformações | padrão do Ultralytics. Os valores efetivos ficam em `resultados/yolo/yolo_cls_256/args.yaml` |

**O que a célula faz**

1. Monta `dados/yolo/` (`train`, `val`, `test`, uma pasta por classe) com **links simbólicos** a partir do `particoes.csv`, sem copiar imagens. O teste fica separado e não é usado no treino.
2. Treina com os pesos pré-treinados.
3. Calcula acurácia e **F1 macro na validação** com o melhor checkpoint.

**Observações**

- O Ultralytics escolhe o `best.pt` pelo critério de acurácia que ele mesmo calcula na validação, e não pelo F1 macro. O F1 macro do `best.pt` é calculado no passo 3, e é ele que entra na comparação.
- O pré-treinamento é permitido **apenas no bônus**. As seis execuções obrigatórias partem de pesos novos.
- A avaliação no **teste** vem depois, uma única vez, e fica fora desta célula.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from ultralytics import YOLO

# --- Configuração (registrar no artigo) ---
RAIZ = Path.cwd().parent                                   # o notebook fica em notebooks/
PESOS = RAIZ / "modelos" / "yolo26n-cls.pt"                # pesos pré-treinados no ImageNet (Ultralytics)
IMGSZ, EPOCAS, LOTE, PACIENCIA, SEMENTE = 256, 50, 32, 10, 42
DESTINO = RAIZ / "dados" / "yolo"                          # train/ val/ test/ com uma pasta por classe
SAIDA = RAIZ / "resultados" / "yolo"

# --- 1. Monta as pastas a partir das MESMAS partições dos outros modelos (links simbólicos, sem copiar) ---
particoes = pd.read_csv(RAIZ / "dados" / "splits" / "particoes.csv")
pasta = {"treino": "train", "validacao": "val", "teste": "test"}
for _, linha in particoes.iterrows():
    origem = RAIZ / linha["caminho_relativo"]
    destino = DESTINO / pasta[linha["particao"]] / linha["classe"] / origem.name
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        os.symlink(origem, destino)

# --- 2. Ajuste fino do YOLO em modo classify (o teste NÃO é usado aqui) ---
PESOS.parent.mkdir(exist_ok=True)
modelo = YOLO(str(PESOS))
modelo.train(data=str(DESTINO), epochs=EPOCAS, imgsz=IMGSZ, batch=LOTE, patience=PACIENCIA,
             seed=SEMENTE, deterministic=True, project=str(SAIDA), name=f"yolo_cls_{IMGSZ}", exist_ok=True)

# --- 3. F1 macro e acurácia na VALIDAÇÃO com o melhor checkpoint (métrica de seleção do projeto) ---
melhor = YOLO(str(SAIDA / f"yolo_cls_{IMGSZ}" / "weights" / "best.pt"))
val = particoes[particoes["particao"] == "validacao"].sort_values("caminho_relativo")
previsoes = melhor.predict([str(RAIZ / c) for c in val["caminho_relativo"]], imgsz=IMGSZ, batch=LOTE, verbose=False)
previstas = [melhor.names[r.probs.top1] for r in previsoes]
print(f"Validação | acurácia {accuracy_score(val['classe'], previstas):.4f} | "
      f"F1 macro {f1_score(val['classe'], previstas, average='macro'):.4f}")